# 04 - Clasificación: Conductores de Alto Riesgo

**Pregunta de negocio:** ¿Podemos identificar conductores de alto riesgo?

## Objetivos
- Construir un clasificador binario para identificar conductores agresivos/de alto riesgo
- Comparar Regresión Logística, Random Forest y XGBoost
- Evaluar con métricas apropiadas: precisión, recall, F1, AUC-ROC
- Optimizar el umbral de decisión según el contexto de negocio
- Seleccionar el mejor modelo y guardarlo para producción

## Teoría: Clasificación Supervisada

### Modelos
- **Regresión Logística**: modelo lineal que predice probabilidad P(Y=1|X) usando la función sigmoide. Simple, interpretable, buen baseline.
  - σ(z) = 1 / (1 + e^(-z)), donde z = β₀ + β₁x₁ + ... + βₚxₚ
- **Árboles de Decisión**: particionan el espacio de features recursivamente. Intuitivos pero propensos a overfitting.
- **Random Forest**: ensemble de árboles entrenados con bootstrap + subsampling de features. Reduce varianza, robusto.
- **XGBoost**: boosting de árboles. Construye árboles secuencialmente, cada uno corrigiendo los errores del anterior. Muy poderoso.

### Métricas de Clasificación
- **Precisión (Precision)**: de los que predije positivos, ¿cuántos realmente lo son? → TP / (TP + FP)
- **Recall (Sensibilidad)**: de los que realmente son positivos, ¿cuántos detecté? → TP / (TP + FN)
- **F1-Score**: media armónica de precisión y recall → 2 · (P · R) / (P + R)
- **AUC-ROC**: área bajo la curva ROC. Mide la capacidad discriminativa del modelo independiente del umbral.

### Matriz de Confusión
| | Predicho Positivo | Predicho Negativo |
|---|---|---|
| **Real Positivo** | TP (Verdadero Positivo) | FN (Falso Negativo) |
| **Real Negativo** | FP (Falso Positivo) | TN (Verdadero Negativo) |

### Ajuste de Umbral
- Por defecto, clasificamos como positivo si P(Y=1) > 0.5
- Pero el umbral óptimo depende del **costo relativo** de errores:
  - Si FN es costoso (no detectar un conductor peligroso) → bajar umbral → más recall
  - Si FP es costoso (penalizar injustamente a un buen conductor) → subir umbral → más precisión

In [ ]:
import os
import glob
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score, f1_score,
    accuracy_score, precision_score, recall_score, roc_auc_score
)

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier as XGBClassifier
    HAS_XGBOOST = False
    print("XGBoost no disponible, usando GradientBoostingClassifier como alternativa")

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
data_dir = os.path.join(project_root, "data/raw")
processed_dir = os.path.join(project_root, "data/processed")
models_dir = os.path.join(project_root, "models")
os.makedirs(models_dir, exist_ok=True)
rng = np.random.default_rng(42)

vtype_colors = {'electrico': '#2ecc71', 'gasolina': '#3498db', 'hibrido': '#9b59b6', 'deportivo': '#e74c3c'}

print(f"XGBoost nativo: {HAS_XGBOOST}")
print("Librerías cargadas correctamente")

## 1. Carga de datos y creación del target

Definimos **conductor agresivo** como aquel en el **top 20%** de:
- `harsh_braking_count`: número de frenadas bruscas
- `speed_std`: variabilidad de velocidad (indicador de conducción errática)

Un conductor es "agresivo" si supera el percentil 80 en **ambas** métricas.

In [ ]:
# Cargar datos procesados
features_path = os.path.join(processed_dir, "features_ml.csv")
merged_path = os.path.join(processed_dir, "vehicle_survey_merged.csv")

if os.path.exists(features_path):
    df = pd.read_csv(features_path)
    print(f"Cargado features_ml.csv: {df.shape}")
else:
    # Fallback: construir features desde telemetría
    print("features_ml.csv no encontrado, construyendo desde telemetría...")
    files = sorted(glob.glob(os.path.join(data_dir, "telemetry/telemetry_*.csv")))
    telemetry = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    fleet = pd.read_csv(os.path.join(data_dir, "fleet_profiles.csv"))
    telemetry = telemetry.merge(fleet[['vehicle_id', 'vehicle_type']], on='vehicle_id', how='left')
    
    df = telemetry.groupby('vehicle_id').agg(
        speed_mean=('speed_kmh', 'mean'),
        speed_std=('speed_kmh', 'std'),
        speed_max=('speed_kmh', 'max'),
        consumption_mean=('fuel_consumption_rate', 'mean'),
        battery_temp_mean=('battery_temp_c', 'mean'),
        battery_soc_mean=('battery_soc_pct', 'mean'),
        motor_rpm_mean=('motor_rpm', 'mean'),
        motor_power_mean=('motor_power_kw', 'mean'),
        acceleration_mean=('acceleration_ms2', 'mean'),
        acceleration_std=('acceleration_ms2', 'std'),
        n_trips=('trip_id', 'nunique'),
        vehicle_type=('vehicle_type', 'first'),
    ).reset_index()
    
    # Crear harsh_braking_count como proxy
    harsh = telemetry[telemetry['acceleration_ms2'] < -3].groupby('vehicle_id').size().reset_index(name='harsh_braking_count')
    df = df.merge(harsh, on='vehicle_id', how='left')
    df['harsh_braking_count'] = df['harsh_braking_count'].fillna(0)
    print(f"Dataset construido: {df.shape}")

print(f"\nColumnas: {list(df.columns)}")
print(f"\nPrimeras filas:")
df.head()

In [ ]:
# Crear variable target: conductor agresivo (top 20% en harsh_braking + speed_std)
# Verificar qué columnas tenemos para el target
risk_cols = []
if 'harsh_braking_count' in df.columns:
    risk_cols.append('harsh_braking_count')
if 'speed_std' in df.columns:
    risk_cols.append('speed_std')
if 'acceleration_std' in df.columns and 'speed_std' not in df.columns:
    risk_cols.append('acceleration_std')

print(f"Columnas disponibles para definir riesgo: {risk_cols}")

# Calcular percentil 80 para cada variable de riesgo
thresholds = {}
for col in risk_cols:
    thresholds[col] = df[col].quantile(0.80)
    print(f"  {col}: percentil 80 = {thresholds[col]:.2f}")

# Target: agresivo si está en top 20% en TODAS las variables de riesgo
if len(risk_cols) >= 2:
    conditions = [df[col] >= thresholds[col] for col in risk_cols]
    df['aggressive_driver'] = (conditions[0] & conditions[1]).astype(int)
else:
    df['aggressive_driver'] = (df[risk_cols[0]] >= thresholds[risk_cols[0]]).astype(int)

# Balance de clases
balance = df['aggressive_driver'].value_counts()
balance_pct = df['aggressive_driver'].value_counts(normalize=True) * 100

print(f"\nBalance de clases:")
print(f"  No agresivo (0): {balance[0]:,} ({balance_pct[0]:.1f}%)")
print(f"  Agresivo (1):    {balance[1]:,} ({balance_pct[1]:.1f}%)")

# Visualizar balance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barplot
colors_bar = ['#2ecc71', '#e74c3c']
axes[0].bar(['No agresivo (0)', 'Agresivo (1)'], balance.values, color=colors_bar, edgecolor='white')
for i, v in enumerate(balance.values):
    axes[0].text(i, v + 0.5, f'{v:,}\n({balance_pct.values[i]:.1f}%)', ha='center', fontsize=12)
axes[0].set_ylabel('Número de conductores')
axes[0].set_title('Distribución del target: Conductor agresivo', fontsize=13, fontweight='bold')

# Scatter de las variables que definen el target
if len(risk_cols) >= 2:
    for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
        subset = df[df['aggressive_driver'] == label]
        axes[1].scatter(subset[risk_cols[0]], subset[risk_cols[1]], 
                       alpha=0.4, s=20, c=color, 
                       label='Agresivo' if label == 1 else 'No agresivo')
    axes[1].axvline(thresholds[risk_cols[0]], color='gray', ls='--', alpha=0.7, label=f'P80 {risk_cols[0]}')
    axes[1].axhline(thresholds[risk_cols[1]], color='gray', ls=':', alpha=0.7, label=f'P80 {risk_cols[1]}')
    axes[1].set_xlabel(risk_cols[0])
    axes[1].set_ylabel(risk_cols[1])
    axes[1].legend()
    axes[1].set_title('Zona de alto riesgo (cuadrante superior derecho)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n→ La intersección del top 20% en ambas variables define conductores agresivos")
print("→ Esto genera un desbalance moderado, apropiado para clasificación binaria")

In [ ]:
# Preparar features y target
# Excluir columnas no predictivas y el target
exclude_cols = ['vehicle_id', 'trip_id', 'aggressive_driver'] + risk_cols
cat_cols = [c for c in df.select_dtypes(include='object').columns if c not in exclude_cols]

# Codificar categóricas
df_model = df.copy()
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    label_encoders[col] = le

feature_cols = [c for c in df_model.columns if c not in exclude_cols]
X = df_model[feature_cols].fillna(0)
y = df_model['aggressive_driver']

print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"Tamaño X: {X.shape}")
print(f"Distribución y: {y.value_counts().to_dict()}")

# Split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Escalar para regresión logística
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTrain: {X_train.shape[0]} muestras")
print(f"Test:  {X_test.shape[0]} muestras")
print(f"Balance train: {y_train.value_counts(normalize=True).to_dict()}")
print(f"Balance test:  {y_test.value_counts(normalize=True).to_dict()}")

## 2. Regresión Logística

Nuestro **baseline**: modelo lineal que estima la probabilidad de ser conductor agresivo.
Ventaja principal: los coeficientes son directamente interpretables.

In [ ]:
# Regresión Logística
lr = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

print("=" * 60)
print("REGRESIÓN LOGÍSTICA - Reporte de Clasificación")
print("=" * 60)
print(classification_report(y_test, y_pred_lr, target_names=['No agresivo', 'Agresivo']))
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob_lr):.4f}")

In [ ]:
# Visualización: Matriz de confusión, ROC, Precision-Recall
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# 1. Matriz de confusión como heatmap
cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No agresivo', 'Agresivo'],
            yticklabels=['No agresivo', 'Agresivo'])
axes[0].set_xlabel('Predicción')
axes[0].set_ylabel('Real')
axes[0].set_title('Matriz de Confusión\nRegresión Logística', fontsize=13, fontweight='bold')

# 2. Curva ROC
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
auc_lr = auc(fpr_lr, tpr_lr)
axes[1].plot(fpr_lr, tpr_lr, color='#3498db', lw=2, label=f'Log. Reg. (AUC={auc_lr:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Aleatorio (AUC=0.500)')
axes[1].fill_between(fpr_lr, tpr_lr, alpha=0.1, color='#3498db')
axes[1].set_xlabel('Tasa de Falsos Positivos (FPR)')
axes[1].set_ylabel('Tasa de Verdaderos Positivos (TPR)')
axes[1].set_title('Curva ROC', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower right')

# 3. Curva Precision-Recall
prec_lr, rec_lr, _ = precision_recall_curve(y_test, y_prob_lr)
ap_lr = average_precision_score(y_test, y_prob_lr)
axes[2].plot(rec_lr, prec_lr, color='#e74c3c', lw=2, label=f'Log. Reg. (AP={ap_lr:.3f})')
axes[2].axhline(y_test.mean(), color='gray', ls='--', alpha=0.5, label=f'Baseline ({y_test.mean():.3f})')
axes[2].fill_between(rec_lr, prec_lr, alpha=0.1, color='#e74c3c')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precisión')
axes[2].set_title('Curva Precisión-Recall', fontsize=13, fontweight='bold')
axes[2].legend(loc='upper right')

plt.suptitle('Regresión Logística: Evaluación Completa', fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print("→ La curva ROC mide la capacidad discriminativa general del modelo")
print("→ La curva Precision-Recall es más informativa cuando hay desbalance de clases")
print("→ AP (Average Precision) resume la curva PR en un solo número")

## 3. Random Forest con GridSearchCV

Random Forest es un ensemble de árboles de decisión. Vamos a optimizar sus hiperparámetros
usando búsqueda en grilla con validación cruzada estratificada.

In [ ]:
# Random Forest con GridSearchCV
rf_base = RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1)

# Grid pequeño pero informativo
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [5, 10],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    rf_base, param_grid, cv=cv, scoring='f1',
    n_jobs=-1, verbose=1, return_train_score=True
)
grid_search.fit(X_train, y_train)

print(f"\nMejores hiperparámetros: {grid_search.best_params_}")
print(f"Mejor F1 (CV): {grid_search.best_score_:.4f}")

# Resultados del grid search
results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df.sort_values('rank_test_score')
print(f"\nTop 5 combinaciones:")
print(results_df[['params', 'mean_test_score', 'std_test_score', 'mean_train_score']].head().to_string())

In [ ]:
# Evaluar mejor Random Forest
rf_best = grid_search.best_estimator_
y_pred_rf = rf_best.predict(X_test)
y_prob_rf = rf_best.predict_proba(X_test)[:, 1]

print("=" * 60)
print("RANDOM FOREST (OPTIMIZADO) - Reporte de Clasificación")
print("=" * 60)
print(classification_report(y_test, y_pred_rf, target_names=['No agresivo', 'Agresivo']))
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob_rf):.4f}")

In [ ]:
# Feature importance del Random Forest
importances = rf_best.feature_importances_
feat_imp = pd.DataFrame({
    'Feature': feature_cols,
    'Importancia': importances
}).sort_values('Importancia', ascending=True)

fig, ax = plt.subplots(figsize=(10, max(6, len(feature_cols) * 0.4)))
colors_imp = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(feat_imp)))
ax.barh(feat_imp['Feature'], feat_imp['Importancia'], color=colors_imp)
ax.set_xlabel('Importancia (Gini)', fontsize=12)
ax.set_title('Random Forest: Importancia de Variables', fontsize=14, fontweight='bold')

# Anotar valores
for i, (_, row) in enumerate(feat_imp.iterrows()):
    ax.text(row['Importancia'] + 0.002, i, f"{row['Importancia']:.3f}", va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\nTop 5 features más importantes:")
for _, row in feat_imp.tail(5).iloc[::-1].iterrows():
    print(f"  → {row['Feature']}: {row['Importancia']:.4f}")

## 4. XGBoost

XGBoost (eXtreme Gradient Boosting) construye árboles secuencialmente, donde cada nuevo árbol
intenta corregir los errores del modelo acumulado. Suele dar los mejores resultados en datos tabulares.

In [ ]:
# XGBoost
if HAS_XGBOOST:
    xgb = XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        scale_pos_weight=balance[0] / balance[1],  # Manejar desbalance
        random_state=42, eval_metric='logloss', verbosity=0
    )
else:
    xgb = XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        random_state=42
    )

xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

print("=" * 60)
print("XGBOOST - Reporte de Clasificación")
print("=" * 60)
print(classification_report(y_test, y_pred_xgb, target_names=['No agresivo', 'Agresivo']))
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob_xgb):.4f}")

In [ ]:
# Comparación de curvas ROC de los 3 modelos
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb)
auc_rf = auc(fpr_rf, tpr_rf)
auc_xgb = auc(fpr_xgb, tpr_xgb)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Curvas ROC
ax = axes[0]
ax.plot(fpr_lr, tpr_lr, color='#3498db', lw=2.5, label=f'Logística (AUC={auc_lr:.3f})')
ax.plot(fpr_rf, tpr_rf, color='#2ecc71', lw=2.5, label=f'Random Forest (AUC={auc_rf:.3f})')
ax.plot(fpr_xgb, tpr_xgb, color='#e74c3c', lw=2.5, label=f'XGBoost (AUC={auc_xgb:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('Tasa de Falsos Positivos (FPR)', fontsize=12)
ax.set_ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=12)
ax.set_title('Curvas ROC: Comparación de Modelos', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

# Curvas Precision-Recall
ax = axes[1]
prec_rf, rec_rf, _ = precision_recall_curve(y_test, y_prob_rf)
prec_xgb, rec_xgb, _ = precision_recall_curve(y_test, y_prob_xgb)
ap_rf = average_precision_score(y_test, y_prob_rf)
ap_xgb = average_precision_score(y_test, y_prob_xgb)

ax.plot(rec_lr, prec_lr, color='#3498db', lw=2.5, label=f'Logística (AP={ap_lr:.3f})')
ax.plot(rec_rf, prec_rf, color='#2ecc71', lw=2.5, label=f'Random Forest (AP={ap_rf:.3f})')
ax.plot(rec_xgb, prec_xgb, color='#e74c3c', lw=2.5, label=f'XGBoost (AP={ap_xgb:.3f})')
ax.axhline(y_test.mean(), color='gray', ls='--', alpha=0.5, label=f'Baseline ({y_test.mean():.3f})')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precisión', fontsize=12)
ax.set_title('Curvas Precisión-Recall: Comparación', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print("→ El modelo con mayor AUC-ROC tiene mejor capacidad discriminativa general")
print("→ El modelo con mayor AP es mejor para detectar la clase minoritaria (agresivos)")

## 5. Ajuste de umbral de decisión

El umbral por defecto es 0.5, pero no siempre es óptimo. En nuestro contexto:

- **Falso Negativo** (no detectar un conductor agresivo): costo ALTO → posible accidente, seguro costoso
- **Falso Positivo** (etiquetar como agresivo a un buen conductor): costo BAJO → alerta innecesaria, molestia menor

→ Preferimos **mayor recall** (detectar más agresivos), a costa de algo de precisión.

In [ ]:
# Análisis de umbrales: usar el mejor modelo hasta ahora
# Determinar cuál es el mejor por AUC
aucs = {'Logística': auc_lr, 'Random Forest': auc_rf, 'XGBoost': auc_xgb}
best_model_name = max(aucs, key=aucs.get)
print(f"Mejor modelo por AUC-ROC: {best_model_name} ({aucs[best_model_name]:.4f})")

# Seleccionar las probabilidades del mejor modelo
if best_model_name == 'Logística':
    y_prob_best = y_prob_lr
elif best_model_name == 'Random Forest':
    y_prob_best = y_prob_rf
else:
    y_prob_best = y_prob_xgb

# Calcular métricas para diferentes umbrales
thresholds_range = np.arange(0.1, 0.91, 0.01)
metrics_by_threshold = []

for t in thresholds_range:
    y_pred_t = (y_prob_best >= t).astype(int)
    if y_pred_t.sum() == 0 or y_pred_t.sum() == len(y_pred_t):
        continue
    metrics_by_threshold.append({
        'Umbral': t,
        'Precisión': precision_score(y_test, y_pred_t, zero_division=0),
        'Recall': recall_score(y_test, y_pred_t, zero_division=0),
        'F1': f1_score(y_test, y_pred_t, zero_division=0),
        'Accuracy': accuracy_score(y_test, y_pred_t),
    })

thresh_df = pd.DataFrame(metrics_by_threshold)

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Métricas vs umbral
ax = axes[0]
ax.plot(thresh_df['Umbral'], thresh_df['Precisión'], color='#3498db', lw=2.5, label='Precisión')
ax.plot(thresh_df['Umbral'], thresh_df['Recall'], color='#e74c3c', lw=2.5, label='Recall')
ax.plot(thresh_df['Umbral'], thresh_df['F1'], color='#2ecc71', lw=2.5, label='F1-Score')
ax.plot(thresh_df['Umbral'], thresh_df['Accuracy'], color='#9b59b6', lw=2, ls='--', label='Accuracy')

# Marcar umbral por defecto y F1 máximo
ax.axvline(0.5, color='gray', ls='--', alpha=0.5, label='Umbral default (0.5)')
best_f1_idx = thresh_df['F1'].idxmax()
best_threshold = thresh_df.loc[best_f1_idx, 'Umbral']
ax.axvline(best_threshold, color='gold', ls='-', lw=2, alpha=0.8, label=f'Mejor F1 (t={best_threshold:.2f})')

ax.set_xlabel('Umbral de decisión', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title(f'Métricas vs Umbral ({best_model_name})', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)

# Panel 2: Tradeoff precisión vs recall con contexto de negocio
ax = axes[1]
ax.plot(thresh_df['Recall'], thresh_df['Precisión'], color='#e74c3c', lw=2.5)
ax.scatter(thresh_df.loc[thresh_df['Umbral'].sub(0.5).abs().idxmin(), 'Recall'],
           thresh_df.loc[thresh_df['Umbral'].sub(0.5).abs().idxmin(), 'Precisión'],
           s=200, c='gray', marker='*', zorder=5, label='t=0.5')
ax.scatter(thresh_df.loc[best_f1_idx, 'Recall'],
           thresh_df.loc[best_f1_idx, 'Precisión'],
           s=200, c='gold', marker='*', zorder=5, label=f't={best_threshold:.2f} (mejor F1)')

# Zona de negocio: preferimos alto recall
ax.axvspan(0.8, 1.0, alpha=0.1, color='green', label='Zona ideal (alto recall)')

ax.set_xlabel('Recall (detectar agresivos)', fontsize=12)
ax.set_ylabel('Precisión (evitar falsas alarmas)', fontsize=12)
ax.set_title('Tradeoff Precisión-Recall', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"\nUmbral por defecto (0.50):")
row_default = thresh_df.loc[thresh_df['Umbral'].sub(0.5).abs().idxmin()]
print(f"  Precisión={row_default['Precisión']:.3f}, Recall={row_default['Recall']:.3f}, F1={row_default['F1']:.3f}")

print(f"\nUmbral óptimo por F1 ({best_threshold:.2f}):")
row_best = thresh_df.loc[best_f1_idx]
print(f"  Precisión={row_best['Precisión']:.3f}, Recall={row_best['Recall']:.3f}, F1={row_best['F1']:.3f}")

print("\n→ En contexto de seguridad vial, un recall alto es más importante que la precisión")
print("→ Es preferible investigar una falsa alarma que no detectar a un conductor peligroso")

## 6. Comparación final de modelos

In [ ]:
# Tabla resumen de todos los modelos
models_summary = []

for name, y_pred, y_prob in [
    ('Regresión Logística', y_pred_lr, y_prob_lr),
    ('Random Forest', y_pred_rf, y_prob_rf),
    ('XGBoost', y_pred_xgb, y_prob_xgb)
]:
    models_summary.append({
        'Modelo': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precisión (agresivo)': precision_score(y_test, y_pred, zero_division=0),
        'Recall (agresivo)': recall_score(y_test, y_pred, zero_division=0),
        'F1 (agresivo)': f1_score(y_test, y_pred, zero_division=0),
        'AUC-ROC': roc_auc_score(y_test, y_prob),
        'AP': average_precision_score(y_test, y_prob),
    })

summary_df = pd.DataFrame(models_summary)
print("COMPARACIÓN DE MODELOS - Conductor de Alto Riesgo")
print("=" * 80)
print(summary_df.round(4).to_string(index=False))

# Mejor modelo
best_idx = summary_df['AUC-ROC'].idxmax()
print(f"\n★ Mejor modelo por AUC-ROC: {summary_df.loc[best_idx, 'Modelo']}")
print(f"  AUC-ROC = {summary_df.loc[best_idx, 'AUC-ROC']:.4f}")
print(f"  F1 = {summary_df.loc[best_idx, 'F1 (agresivo)']:.4f}")

In [ ]:
# Visualización de comparación
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Barplot de métricas
ax = axes[0]
metrics_to_plot = ['Precisión (agresivo)', 'Recall (agresivo)', 'F1 (agresivo)', 'AUC-ROC']
x = np.arange(len(metrics_to_plot))
width = 0.25

model_colors = ['#3498db', '#2ecc71', '#e74c3c']
for i, (_, row) in enumerate(summary_df.iterrows()):
    values = [row[m] for m in metrics_to_plot]
    ax.bar(x + i * width, values, width, label=row['Modelo'], color=model_colors[i], alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(['Precisión', 'Recall', 'F1', 'AUC-ROC'])
ax.set_ylabel('Score')
ax.set_title('Comparación de Modelos por Métrica', fontsize=14, fontweight='bold')
ax.legend()
ax.set_ylim(0, 1.1)

# Matrices de confusión lado a lado
ax = axes[1]
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
all_cms = [cm_lr, confusion_matrix(y_test, y_pred_rf), cm_xgb]
cm_names = ['Log. Reg.', 'Random Forest', 'XGBoost']

cell_text = []
for metric_name in ['TN', 'FP', 'FN', 'TP']:
    row_vals = []
    for cm in all_cms:
        if metric_name == 'TN': row_vals.append(cm[0, 0])
        elif metric_name == 'FP': row_vals.append(cm[0, 1])
        elif metric_name == 'FN': row_vals.append(cm[1, 0])
        else: row_vals.append(cm[1, 1])
    cell_text.append(row_vals)

table = ax.table(cellText=cell_text, rowLabels=['TN', 'FP', 'FN', 'TP'],
                 colLabels=cm_names, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1, 2)
ax.axis('off')
ax.set_title('Matrices de Confusión', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Guardar el mejor modelo

In [ ]:
# Seleccionar y guardar el mejor modelo
best_model_final = summary_df.loc[summary_df['AUC-ROC'].idxmax(), 'Modelo']

if best_model_final == 'Regresión Logística':
    model_to_save = lr
elif best_model_final == 'Random Forest':
    model_to_save = rf_best
else:
    model_to_save = xgb

# Guardar modelo, scaler y metadata
model_path = os.path.join(models_dir, "classification_driver_risk.pkl")
model_package = {
    'model': model_to_save,
    'model_name': best_model_final,
    'scaler': scaler if best_model_final == 'Regresión Logística' else None,
    'feature_cols': feature_cols,
    'label_encoders': label_encoders,
    'optimal_threshold': best_threshold,
    'metrics': summary_df.loc[summary_df['AUC-ROC'].idxmax()].to_dict(),
    'risk_definition': f"Top 20% en {risk_cols}",
}

joblib.dump(model_package, model_path)
print(f"Modelo guardado en: {model_path}")
print(f"Modelo seleccionado: {best_model_final}")
print(f"Tamaño del archivo: {os.path.getsize(model_path) / 1024:.1f} KB")

# Verificar que se puede cargar
loaded = joblib.load(model_path)
print(f"\nVerificación - Modelo cargado: {loaded['model_name']}")
print(f"Features: {loaded['feature_cols']}")
print(f"Umbral óptimo: {loaded['optimal_threshold']:.2f}")

## Resumen

### Hallazgos clave

**1. Definición del target:**
- Conductor agresivo = top 20% en frenadas bruscas Y variabilidad de velocidad
- Esto genera un desbalance moderado que manejamos con `class_weight='balanced'`

**2. Comparación de modelos:**
- Los tres modelos logran discriminar conductores de alto riesgo
- Los modelos de ensemble (RF, XGBoost) superan a la regresión logística en AUC-ROC
- XGBoost generalmente ofrece el mejor balance precisión/recall

**3. Ajuste de umbral:**
- El umbral por defecto (0.5) no es óptimo para nuestro contexto de seguridad vial
- Un umbral más bajo aumenta el recall → detectamos más conductores peligrosos
- El costo de un falso negativo (accidente no prevenido) >> costo de falso positivo (alerta innecesaria)

### Implicaciones de negocio
- → **Alertas proactivas**: identificar conductores de alto riesgo antes de que ocurra un incidente
- → **Seguros personalizados**: ajustar primas según el perfil de riesgo
- → **Programas de capacitación**: dirigir recursos a conductores con mayor probabilidad de riesgo
- → **Monitoreo en tiempo real**: usar las features del modelo para dashboards operativos

### Siguiente notebook
→ `05_classification_churn_risk.ipynb`: Clasificación de riesgo de abandono de clientes, con técnicas para desbalance severo (SMOTE)